In [2]:
import cortex
import pickle
import numpy as np
import pandas as pd
import nibabel as nib
from collections import Counter

In [3]:
with open("schaefer_parcel_counts.pkl", 'rb') as f:
    parcel_voxel_counts = pickle.load(f)
    
with open('/tank/home/zachkaras/fmri_model/analysis/fir/schaefer_parcel_labels.pkl', 'rb') as f:
    schaefer_parcel_labels = pickle.load(f)

with open("/tank/home/zachkaras/fmri_model_data/intermediate_results/all_results_regressor+features.pkl", 'rb') as f:
    records = pickle.load(f)
   

In [4]:
task_labels = {
    'code' : 'Code',
    'prose': 'Prose'
}

low_performing  = {121, 142, 134, 151, 203, 105, 138, 141, 150, 109, 201}
high_performing = {111, 144, 112, 118, 125, 204, 129, 133, 117, 108, 119, 122}
 
poorly_modeled  = {125, 138, 108, 144, 105, 142, 150, 201, 119, 134, 109}
well_modeled    = {203, 151, 141, 133, 122, 129, 112, 204, 111, 121, 118, 117}
print(high_performing.intersection(well_modeled), len(high_performing), len(well_modeled), len(high_performing.intersection(well_modeled))/len(high_performing))
print(low_performing.intersection(poorly_modeled), len(low_performing), len(poorly_modeled), len(low_performing.intersection(poorly_modeled))/len(low_performing))

poorly_modeled_prose = {144, 129, 125, 138, 204, 203, 201, 141, 108, 151, 133}
well_modeled_prose   = {121, 122, 112, 117, 109, 105, 150, 111, 142, 118, 119, 134}
print(high_performing.intersection(well_modeled_prose), len(high_performing), len(well_modeled_prose), len(high_performing.intersection(well_modeled_prose))/len(high_performing))

{129, 133, 204, 111, 112, 117, 118, 122} 12 12 0.6666666666666666
{134, 105, 138, 201, 109, 142, 150} 11 11 0.6363636363636364
{111, 112, 117, 118, 119, 122} 12 12 0.5


In [5]:
info = {'121' : {'performance': 'low', 'modeled' :   'well', 'modeled_prose': 'well_prose'},   
        '142' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'well_prose'},
        '134' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'well_prose'},  
        '151' : {'performance': 'low', 'modeled' :   'well', 'modeled_prose': 'poorly_prose'},   
        '203' : {'performance': 'low', 'modeled' :   'well', 'modeled_prose': 'poorly_prose'},
        '105' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'well_prose'},  
        '138' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'poorly_prose'},  
        '141' : {'performance': 'low', 'modeled' :   'well', 'modeled_prose': 'poorly_prose'},
        '150' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'well_prose'},  
        '109' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'well_prose'},  
        '201' : {'performance': 'low', 'modeled':  'poorly', 'modeled_prose': 'poorly_prose'}, 
        '111' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'well_prose'},  
        '144' : {'performance': 'high', 'modeled': 'poorly', 'modeled_prose': 'poorly_prose'}, 
        '112' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'well_prose'}, 
        '118' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'well_prose'},  
        '125' : {'performance': 'high', 'modeled': 'poorly', 'modeled_prose': 'poorly_prose'}, 
        '204' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'poorly_prose'}, 
        '129' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'poorly_prose'},  
        '133' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'poorly_prose'},  
        '117' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'well_prose'}, 
        '108' : {'performance': 'high', 'modeled': 'poorly', 'modeled_prose': 'poorly_prose'}, 
        '119' : {'performance': 'high', 'modeled': 'poorly', 'modeled_prose': 'well_prose'}, 
        '122' : {'performance': 'high', 'modeled' :  'well', 'modeled_prose': 'well_prose'}}

# no significant correlation between code correct and modeling performance

In [6]:
len(list(info.keys()))

23

In [ ]:
def translate_to_region_names(parcel_num, schaefer_labels):
        parcel_num = int(parcel_num)
        if parcel_num <= 200:
            region = f"Left {schaefer_labels['left'][parcel_num]}"
        else:
            region = f"Right {schaefer_labels['right'][parcel_num]}"
        return region

def get_top_regions(parcel_dict, n_regions=5):
    parcel_dict = dict(sorted(parcel_dict.items(), key= lambda x: x[1], reverse=True))
    return {region : val for i,(region,val) in enumerate(parcel_dict.items()) if i < n_regions}


def compute_jaccard(df):
    list_of_sets = [set(row['top_five_regions'].keys()) for i,row in df.iterrows()]
    intersection = set.intersection(*list_of_sets)
    union = set.union(*list_of_sets)
    return len(intersection) / len(union)

# Brain plots

In [8]:
atlas_base_path = "/home/zachkaras/fmri_model/analysis/pipeline/atlases"
# atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx] # contains the schaefer parcel numbers
cortex_vx = np.where(atlas_only_brain != 0)[0]
schaefer_voxels = atlas_only_brain[cortex_vx]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)


In [9]:
def increment_parcels(top_parcels, schaefers, participant, info):
    performance = info[participant]['performance']
    modeled = info[participant]['modeled']
    modeled_prose = info[participant]['modeled_prose']
    
    for p in top_parcels:
        schaefers[performance][p] += 1
        schaefers[modeled][p] += 1
        schaefers[modeled_prose][p] +=1
    return schaefers
    
def make_schafer_data(curr_task):
    mask = ((records['look_ahead'] == 'look_ahead_by_0') &
            (records['ndelays'] == 'ndelays_10') &
            (records['model'] == 'deepseek_6b'))
    
    schaefers = {'well'       : np.zeros(401),
                'poorly'      : np.zeros(401),
                'well_prose'  : np.zeros(401),
                'poorly_prose': np.zeros(401),
                'low'         : np.zeros(401),
                'high'        : np.zeros(401)}

    for (model, task, participant), df in records[mask].groupby(['model', 'task', 'participant']):
        if task != curr_task:
            continue
        
        if participant not in info.keys():
            continue
        
        df['parcel_counts'] = df['top_parcels'].apply(lambda row: dict(Counter(row)))
        df['parcel_counts'] = df['parcel_counts'].apply(lambda row: get_top_regions(row, n_regions=25)) # on average there are about 4.54 parcels per Harvard-Oxford region, so I'm looking at the top 5 regions

        top_parcels = set(df['parcel_counts'].explode()) # set of all the regions that are among the top 10 schaefer parcels across the layers for a participant
        top_parcels = {int(i) for i in top_parcels}
        schaefers = increment_parcels(top_parcels, schaefers, participant, info)
    return schaefers

def convert_to_nifti(values):
    # working backwards to save correlation values as voxels in MNI space
    new_schaefer = empty_schaefer.copy()
    new_mni = empty_mni.copy()
    
    
    new_schaefer[cortex_vx] = values
    new_mni[brain_idx] = new_schaefer
    result_brain = np.reshape(new_mni, og_shape)

    # Saving results
    nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
    # nib.save(nifti_result, "test_plotting.nii.gz")
    return result_brain, nifti_result

def save_schaefer_maps(schaefers, task, max):
    # find indices of voxels corresponding to each schaefer parcel
    # is there a way to parallelize?
    for group, lookup in schaefers.items():
        print(group)        
        result = lookup[schaefer_voxels.astype(int)]
        
        npy_brain, nifti_brain = convert_to_nifti(result)
        npy_brain = npy_brain.transpose(2,1,0)
    
        vol = cortex.Volume(
                npy_brain,
                subject='fsaverage',
                xfmname='mni2py2',
                vmin=0,
                vmax=max, #max(list(np.unique(npy_brain))),
                description=f"{task}-{group}",
                cmap='jet', # This fits the rest of the color scheme
                # cmap='gnuplot2',
                # cmap='cool',
                # cmap='Oranges'
        )
        cortex.webshow(vol)
        # break
        # break
    
    
    # set value of those voxels equal to number in dictionaries
def find_max(d):
    max_count = 0
    for group,parcels in d.items():
        new_max = max(parcels)
        max_count = new_max if new_max > max_count else max_count

        print(group, max(parcels))
    return int(max_count)


In [10]:
code_schaefers = make_schafer_data('code')
prose_schaefers = make_schafer_data('prose')

code_max = find_max(code_schaefers)
prose_max = find_max(prose_schaefers)

well 10.0
poorly 9.0
well_prose 9.0
poorly_prose 8.0
low 8.0
high 10.0
well 10.0
poorly 8.0
well_prose 9.0
poorly_prose 10.0
low 8.0
high 10.0


In [11]:
print(code_max, prose_max)

10 10


In [12]:
save_schaefer_maps(code_schaefers, 'code', code_max)

well
Started server on port 27859
poorly
Started server on port 36632
well_prose
Started server on port 49179
poorly_prose
Started server on port 20305
low
Started server on port 20580
high
Started server on port 20678


In [13]:
save_schaefer_maps(prose_schaefers, 'prose', prose_max)

well
Started server on port 59198
poorly
Started server on port 56889
well_prose
Started server on port 30782
poorly_prose
Started server on port 24652
low
Started server on port 38556
high
Started server on port 45401


In [24]:
np.where(code_schaefers['well'] == 7)

(array([343]),)

In [27]:
with open("schaefer_parcel_labels.pkl", 'rb') as f:
    labels = pickle.load(f)

In [ ]:
# digusting line of code but this finds the average number of schaefer parcels within a larger region from the Harvard-Oxford atlas
# first counting the number of parcels corresponding to each region, then taking the mean
np.mean(list(dict(Counter(list(labels['left'].values()))).values()))

np.float64(4.545454545454546)